# 01 — Sitemap to HTML: writing XSLT pythonically

**What you learn:**

- An XSLT stylesheet *is* XML, so it renders through the shared `XmlRenderer` — there is **no** XSLT-specific renderer.
- Namespaced instructions (`xsl:value-of`, `xsl:for-each`) come from `@element(_meta={"ns": "xsl", "local": "..."})`: the Python method keeps a legal name (`value_of`), while the emitted tag carries the prefix and hyphen (`xsl:value-of`).
- The `xsl` prefix is declared as a plain attribute on the root: `stylesheet(xmlns_xsl=...)` surfaces as `xmlns:xsl="..."`.
- HTML output tags are *literal result elements* of the XSLT grammar (interleaved with the instructions at the same level), not a nested HTML sub-builder.
- The stylesheet is then applied to a real sitemap with `lxml`; the `{loc}` attribute-value-template is resolved by the XSLT processor at runtime.

**Prerequisites:** none for the XSLT dialect; the sitemap XSD example supplies the input document.

## 1. Define the stylesheet handler

Subclass `XsltBuilderHandler` and build the stylesheet inside `main`. Note how `xsl:*` instructions (`template`, `for_each`, `value_of`) sit right next to literal HTML result elements (`html`, `table`, `tr`). They share one tree — that is exactly how XSLT works.

In [ ]:
from genro_builders.contrib.xslt import XsltBuilderHandler

XSL = "http://www.w3.org/1999/XSL/Transform"


class SitemapToHtml(XsltBuilderHandler):
    """A stylesheet that renders a <urlset> as an HTML table of URLs."""

    def main(self, root):
        ss = root.stylesheet(version="1.0", xmlns_xsl=XSL)
        ss.output(method="html", encoding="UTF-8", indent="yes")

        tpl = ss.template(match="/urlset")
        html = tpl.html()
        html.head().title("Sitemap")
        body = html.body()
        body.h1("Sitemap")

        table = body.table()
        header = table.thead().tr()
        header.th("URL")
        header.th("Last modified")
        header.th("Priority")

        loop = table.tbody().for_each(select="url")
        row = loop.tr()
        row.td().a(href="{loc}").value_of(select="loc")
        row.td().value_of(select="lastmod")
        row.td().value_of(select="priority")

## 2. `create()` — populate the source

`create()` runs `main` and builds the source Bag. Each `xsl:*` node carries its `_meta` (`ns`/`local`); the literal result elements do not.

In [ ]:
sheet = SitemapToHtml()
sheet.create()
print(sheet.source.to_xml())

## 3. `render()` — produce the stylesheet

Rendering rides the shared `XmlRenderer`. Watch the `xsl:` prefixes appear: `value_of` -> `<xsl:value-of>`, `for_each` -> `<xsl:for-each>`, and `xmlns_xsl` -> `xmlns:xsl`. The `{loc}` in `href` is left verbatim — it is an XSLT attribute-value-template, not a builder pointer.

In [ ]:
stylesheet_xml = sheet.render(target=False, doc_header=True, pretty=True)
print(stylesheet_xml)

## 4. Apply the stylesheet to a sitemap

Build a small sitemap with the sitemap XSD dialect, then transform it with `lxml`. The XSLT processor resolves `{loc}` to each URL at runtime.

In [ ]:
from lxml import etree

from genro_builders.contrib.xsd.examples.sitemap import SitemapHandler


class SampleSitemap(SitemapHandler):
    def main(self, root):
        urlset = root.urlset()
        home = urlset.url()
        home.loc("https://www.example.com/")
        home.lastmod("2026-06-01")
        about = urlset.url()
        about.loc("https://www.example.com/about")
        about.lastmod("2026-05-20")


sm = SampleSitemap()
sm.create()
sitemap_xml = sm.render(mode="xml", target=False, doc_header=True)

transform = etree.XSLT(etree.fromstring(stylesheet_xml.encode("utf-8")))
result_html = str(transform(etree.fromstring(sitemap_xml.encode("utf-8"))))
print(result_html)

## 5. Inline preview

Render the transformed HTML inside the notebook.

In [ ]:
from IPython.display import HTML

HTML(result_html)